## Functions, Imports and Loads

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 0 — GLOBAL SETUP (RUN FIRST)
# ══════════════════════════════════════════════════════════════════════════════

import os
import glob
import numpy as np
import pandas as pd

# plotting (for later)
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# stats
from scipy import stats

# reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ─────────────────────────────────────────────────────────────────────────────
# BASE PATH (adjust only if needed)
# ─────────────────────────────────────────────────────────────────────────────
BASE = "/home/jupyter/notebooks/reid_clean"

# ─────────────────────────────────────────────────────────────────────────────
# DATA PATHS
# ─────────────────────────────────────────────────────────────────────────────

# Utility
OASIS_UTILITY = f"{BASE}/full_pipeline/results/oasis/utility/test_results.csv"
NIH_UTILITY   = f"{BASE}/reid_chest-NIH/results/nih_cxr/utility/per_run_results.csv"

# Feature-based Re-ID
OASIS_REID_FE = f"{BASE}/full_pipeline/results/oasis/reid_feature_ext/OASIS_ReID_FeatureExt.csv"
NIH_REID_FE   = f"{BASE}/reid_chest-NIH/results/nih_cxr/reid_feature_ext/NIH_CXR_ReID_FeatureExt.csv"

# Oracle Re-ID
OASIS_REID = f"{BASE}/full_pipeline/results/oasis/reid/OASIS_Complete_Results.csv"
NIH_REID   = f"{BASE}/reid_chest-NIH/results/nih_cxr/reid/NIH_CXR_ReID_Results.csv"

# ─────────────────────────────────────────────────────────────────────────────
# GS CONDITIONS
# ─────────────────────────────────────────────────────────────────────────────
COND_ORDER  = ["raw", "gs0", "gs10", "gs20", "gs30", "gs40", "gs50"]
COND_LABELS = ["Raw", "GS-0%", "GS-10%", "GS-20%", "GS-30%", "GS-40%", "GS-50%"]

MODEL_RENAME = {
    "resnet18": "ResNet-18",
    "densenet121": "DenseNet-121"
}

# ─────────────────────────────────────────────────────────────────────────────
# BOOTSTRAP SETTINGS (CRITICAL — KEEP CONSISTENT)
# ─────────────────────────────────────────────────────────────────────────────
BOOTSTRAP_B = 2000
CI_LOW  = 2.5
CI_HIGH = 97.5

# ─────────────────────────────────────────────────────────────────────────────
# OASIS t-CI SETTINGS (5 folds)
# ─────────────────────────────────────────────────────────────────────────────
T_CRIT_OASIS = stats.t.ppf(0.975, df=4)  # = 2.776

print("✓ Cell 0 loaded")
print(f"BASE PATH: {BASE}")
print(f"Bootstrap B = {BOOTSTRAP_B}, Seed = {RANDOM_SEED}")

✓ Cell 0 loaded
BASE PATH: /home/jupyter/notebooks/reid_clean
Bootstrap B = 2000, Seed = 42


In [2]:
import os
import glob
import pandas as pd

BASE = "/home/jupyter/notebooks/reid_clean"

paths_to_check = {
    "NIH utility results dir": f"{BASE}/reid_chest-NIH/results/nih_cxr/utility",
    "NIH FE ReID dir": f"{BASE}/reid_chest-NIH/results/nih_cxr/reid_feature_ext",
    "NIH oracle ReID dir": f"{BASE}/reid_chest-NIH/results/nih_cxr/reid",
    "NIH reconstruction dirs": f"{BASE}/reid_chest-NIH",
    "OASIS reconstruction dirs": f"{BASE}/full_pipeline/results/oasis",
}

for label, path in paths_to_check.items():
    print("\n" + "="*90)
    print(label)
    print(path)
    print("="*90)
    
    if not os.path.exists(path):
        print("MISSING PATH")
        continue
    
    files = []
    for ext in ["*.csv", "*.pt", "*.pth", "*.pkl", "*.json", "*.npz", "*.npy"]:
        files.extend(glob.glob(os.path.join(path, "**", ext), recursive=True))
    
    print(f"Found {len(files)} files")
    for f in files[:80]:
        size_mb = os.path.getsize(f) / (1024**2)
        print(f"{size_mb:8.2f} MB  {f}")
    
    if len(files) > 80:
        print(f"... {len(files)-80} more files not shown")


NIH utility results dir
/home/jupyter/notebooks/reid_clean/reid_chest-NIH/results/nih_cxr/utility
Found 4 files
    0.06 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/results/nih_cxr/utility/per_run_results.csv
    0.00 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/results/nih_cxr/utility/fullft/fullft_results.csv
    0.00 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/results/nih_cxr/utility/gradcam/gs0_artifact_pixel_stats.csv
    0.38 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/results/nih_cxr/utility/gradcam/containment_ratio_results.csv

NIH FE ReID dir
/home/jupyter/notebooks/reid_clean/reid_chest-NIH/results/nih_cxr/reid_feature_ext
Found 1 files
    0.21 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/results/nih_cxr/reid_feature_ext/NIH_CXR_ReID_FeatureExt.csv

NIH oracle ReID dir
/home/jupyter/notebooks/reid_clean/reid_chest-NIH/results/nih_cxr/reid
Found 1 files
    0.01 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/results/nih

In [3]:
csv_files = []

for root in [
    f"{BASE}/reid_chest-NIH/results/nih_cxr/utility",
    f"{BASE}/reid_chest-NIH/results/nih_cxr/reid_feature_ext",
    f"{BASE}/reid_chest-NIH/results/nih_cxr/reid",
    f"{BASE}/full_pipeline/results/oasis",
]:
    csv_files.extend(glob.glob(os.path.join(root, "**", "*.csv"), recursive=True))

print(f"Total CSV files found: {len(csv_files)}")

for f in csv_files:
    try:
        df = pd.read_csv(f, nrows=5)
        print("\n" + "="*90)
        print(f)
        print("Columns:")
        print(df.columns.tolist())
        print("Shape preview:", df.shape)
        print(df.head(2))
    except Exception as e:
        print("\nFAILED:", f)
        print(e)

Total CSV files found: 44

/home/jupyter/notebooks/reid_clean/reid_chest-NIH/results/nih_cxr/utility/per_run_results.csv
Columns:
['arch', 'init', 'train_cond', 'eval_cond', 'attacker_type', 'eval_split', 'best_epoch', 'best_val_auc', 'macro_auc', 'bal_acc', 'top1_acc', 'male_macro_auc', 'female_macro_auc', 'male_n', 'female_n', 'male_bal_acc', 'female_bal_acc', 'auc_NoFind', 'auc_Infilt', 'auc_Atelec', 'auc_Effus', 'auc_Nodule', 'auc_PneuTx', 'auc_Mass', 'auc_Consol', 'auc_PlThck', 'auc_CardMeg', 'auc_Emphy', 'male_auc_NoFind', 'male_auc_Infilt', 'male_auc_Atelec', 'male_auc_Effus', 'male_auc_Nodule', 'male_auc_PneuTx', 'male_auc_Mass', 'male_auc_Consol', 'male_auc_PlThck', 'male_auc_CardMeg', 'male_auc_Emphy', 'female_auc_NoFind', 'female_auc_Infilt', 'female_auc_Atelec', 'female_auc_Effus', 'female_auc_Nodule', 'female_auc_PneuTx', 'female_auc_Mass', 'female_auc_Consol', 'female_auc_PlThck', 'female_auc_CardMeg', 'female_auc_Emphy']
Shape preview: (5, 50)
       arch        init tra

## Cell 1 — NIH Oracle Re-ID (Clean + Standardized)

### Adaptive

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — NIH ORACLE RE-ID (LOAD + CLEAN)
# ══════════════════════════════════════════════════════════════════════════════

# Load
nih_reid = pd.read_csv(NIH_REID)

print("Raw shape:", nih_reid.shape)
display(nih_reid.head())

# ─────────────────────────────────────────────────────────────────────────────
# Standardize column names
# ─────────────────────────────────────────────────────────────────────────────
df = nih_reid.copy()

df = df.rename(columns={
    "Model": "arch",
    "Initialization": "init",
    "GS_Level": "condition",
    "GS_Percentage": "gs_pct",
    "Top1_Mean": "top1_mean",
    "Top1_Std": "top1_std",
    "Top1_CI_Lower": "ci_lower",
    "Top1_CI_Upper": "ci_upper",
    "Top5_Mean": "top5_mean",
    "pvalue_vs_raw_paired": "p_value"
})

# normalize condition labels
df["condition"] = df["condition"].str.lower().str.replace(" ", "")

# rename models
df["arch"] = df["arch"].map(MODEL_RENAME)

# keep only relevant columns
df = df[[
    "arch", "init", "condition", "gs_pct",
    "top1_mean", "top1_std",
    "ci_lower", "ci_upper",
    "top5_mean", "p_value",
    "N_Repeats"
]]

# sort
df = df.sort_values(["arch", "gs_pct"]).reset_index(drop=True)

print("\n✓ Cleaned NIH Oracle Re-ID:")
display(df)

Raw shape: (28, 25)


,RunGroupID,Model,Initialization,GS_Level,GS_Percentage,Top1_Mean,Top1_Std,Top1_CI_Lower,Top1_CI_Upper,Top5_Mean,...,Top1_GSTrain_RawTest,Top5_GSTrain_RawTest,F1_GSTrain_RawTest,TrueProb_GSTrain_RawTest_Mean,TrueProb_GSTrain_RawTest_Median,AccuracyDrop_pp,AccuracyDrop_relative,pvalue_vs_raw_paired,N_Repeats,PerRun_Top1
0,NIH_CXR__resnet18_Pretrained__NR5__E30__LR0.00...,resnet18,Pretrained,raw,0.0,69.487346,0.244535,69.273001,69.701691,81.719663,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,"[69.37053861129137, 69.63011031797534, 69.6301..."
1,NIH_CXR__resnet18_Pretrained__NR5__E30__LR0.00...,resnet18,Pretrained,gs50,50.0,13.419857,1.746482,11.888998,14.950716,26.742375,...,0.064893,0.324465,0.000006,0.000374,0.0,56.067489,80.687337,3.210993e-07,5,"[16.19078520441272, 11.810512654120702, 12.978..."
2,NIH_CXR__resnet18_Pretrained__NR5__E30__LR0.00...,resnet18,Pretrained,gs40,40.0,18.345230,1.085284,17.393937,19.296524,33.303050,...,0.097339,0.292018,0.000002,0.000366,0.0,51.142116,73.599178,2.222609e-08,5,"[18.364698247890978, 18.624269954574952, 19.66..."
3,NIH_CXR__resnet18_Pretrained__NR5__E30__LR0.00...,resnet18,Pretrained,gs30,30.0,24.646334,1.228817,23.569228,25.723440,40.720311,...,0.097339,0.778715,0.000329,0.000413,0.0,44.841012,64.531192,2.108967e-07,5,"[26.703439325113564, 24.36729396495782, 23.685..."
4,NIH_CXR__resnet18_Pretrained__NR5__E30__LR0.00...,resnet18,Pretrained,gs20,20.0,32.076574,0.575685,31.571964,32.581184,49.175860,...,0.129786,0.713822,0.000365,0.000377,0.0,37.410772,53.838252,2.292971e-08,5,"[32.965606748864374, 31.732641142115508, 32.34..."



✓ Cleaned NIH Oracle Re-ID:


,arch,init,condition,gs_pct,top1_mean,top1_std,ci_lower,ci_upper,top5_mean,p_value,N_Repeats
0,DenseNet-121,Pretrained,raw,0.0,72.420506,1.283380,71.295574,73.545438,83.945490,NaN,5
1,DenseNet-121,Pretrained,gs0,0.0,63.634004,0.841234,62.896630,64.371378,77.637897,2.599935e-04,5
2,DenseNet-121,Scratch,raw,0.0,17.767683,0.406294,17.411551,18.123816,33.945490,NaN,5
3,DenseNet-121,Scratch,gs0,0.0,16.229721,0.354989,15.918560,16.540882,32.057106,2.318038e-04,5
4,DenseNet-121,Pretrained,gs10,10.0,39.857236,0.835016,39.125312,40.589159,56.359507,4.807010e-07,5
5,DenseNet-121,Scratch,gs10,10.0,11.817002,0.730370,11.176805,12.457199,25.645685,7.478263e-05,5
6,DenseNet-121,Pretrained,gs20,20.0,31.083712,0.679829,30.487816,31.679608,47.709280,7.462988e-07,5
7,DenseNet-121,Scratch,gs20,20.0,9.759896,0.504436,9.317739,10.202053,21.174562,3.515681e-05,5
8,DenseNet-121,Pretrained,gs30,30.0,25.230370,0.839731,24.494314,25.966426,42.044127,3.771925e-08,5
9,DenseNet-121,Scratch,gs30,30.0,8.234912,0.872261,7.470342,8.999483,19.513303,5.116421e-05,5


### Blind

In [27]:
import torch
import torch.nn as nn
import torchvision.models as tv_models
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from scipy import stats
import pandas as pd
from collections import Counter
import os

BASE     = "/home/jupyter/notebooks/reid_clean"
SAVE_DIR = f"{BASE}/CI_results"
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
B_BOOT, SEED = 2000, 42
print("Device:", DEVICE)

NIH_REID_MODEL_DIR = f"{BASE}/reid_chest-NIH/models/nih_cxr/reid"
NIH_TEST_NPZ       = f"{BASE}/reid_chest-NIH/data/chest/reid_npz/reid_test_raw.npz"

GS_LEVELS_NIH = {
    'raw' :'raw',
    'gs0' :'gs0',
    'gs10':'gs10',
    'gs20':'gs20',
    'gs30':'gs30',
    'gs40':'gs40',
    'gs50':'gs50',
}

# ── Model builder ──────────────────────────────────────────────────────────
def build_nih_reid_model(arch, gs_str, n_classes):
    fname     = f"{arch}_Pretrained_{gs_str}_best.pth"
    ckpt_path = f"{NIH_REID_MODEL_DIR}/{arch}/{fname}"
    if arch == 'resnet18':
        m = tv_models.resnet18(weights=None)
        m.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        m.fc    = nn.Linear(m.fc.in_features, n_classes)
    else:
        m = tv_models.densenet121(weights=None)
        m.features.conv0 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        m.classifier      = nn.Linear(m.classifier.in_features, n_classes)
    state = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    elif isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    m.load_state_dict(state, strict=True)
    m.eval()
    return m.to(DEVICE), ckpt_path

# ── Inference ──────────────────────────────────────────────────────────────
def get_predictions(model, tensor, batch_size=256):
    ds = TensorDataset(tensor)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for (xb,) in dl:
            preds.append(model(xb.to(DEVICE)).argmax(dim=1).cpu())
    return torch.cat(preds).numpy()

# ── Subject-level Top-1 ────────────────────────────────────────────────────
def subject_top1(preds, labels, subject_ids):
    unique_subj = np.unique(subject_ids)
    correct = 0
    for s in unique_subj:
        idx        = np.where(subject_ids == s)[0]
        true_label = labels[idx[0]]
        vote       = Counter(preds[idx]).most_common(1)[0][0]
        if vote == true_label:
            correct += 1
    return correct / len(unique_subj)

# ── Bootstrap ─────────────────────────────────────────────────────────────
def bootstrap_top1(preds, labels, subject_ids, B=2000, seed=42):
    rng         = np.random.default_rng(seed)
    unique_subj = np.unique(subject_ids)
    point       = subject_top1(preds, labels, subject_ids)
    boot_accs   = []
    for _ in range(B):
        samp = rng.choice(unique_subj, size=len(unique_subj), replace=True)
        idx  = np.concatenate([np.where(subject_ids==s)[0] for s in samp])
        boot_accs.append(
            subject_top1(preds[idx], labels[idx], subject_ids[idx]))
    boot_accs = np.array(boot_accs)
    return (point*100,
            np.percentile(boot_accs, 2.5)*100,
            np.percentile(boot_accs, 97.5)*100)

# ── Load NIH raw test data ─────────────────────────────────────────────────
nih_data     = np.load(NIH_TEST_NPZ, allow_pickle=True)
nih_images   = nih_data['images'].astype(np.float32)
nih_labels   = nih_data['labels']
nih_subjects = nih_data['patient_ids']

# Pool normalization — match training
pool_mean = nih_images.mean()
pool_std  = nih_images.std()
nih_norm  = (nih_images - pool_mean) / (pool_std + 1e-8)
nih_tensor = torch.tensor(nih_norm[:, None, :, :])

N_CLASSES = len(np.unique(nih_labels))
print(f"NIH: {nih_tensor.shape}, "
      f"subjects={len(np.unique(nih_subjects))}, "
      f"n_classes={N_CLASSES}")

# ── Run ────────────────────────────────────────────────────────────────────
results = []
ARCHS   = ['resnet18', 'densenet121']

for arch in ARCHS:
    arch_label = 'ResNet-18' if arch == 'resnet18' else 'DenseNet-121'
    print(f"\n{'='*55}")
    print(f"NIH | {arch_label} | Blind (trained on GS → test on raw)")
    print(f"{'='*55}")

    for cond_key, gs_str in GS_LEVELS_NIH.items():
        try:
            model, ckpt = build_nih_reid_model(arch, gs_str, N_CLASSES)
            preds       = get_predictions(model, nih_tensor)
            pt, lo, hi  = bootstrap_top1(
                preds, nih_labels, nih_subjects, B_BOOT, SEED)
            print(f"  {cond_key:6s}: {pt:.2f}% [{lo:.2f}, {hi:.2f}]  "
                  f"← {ckpt.split('/')[-1]}")
            results.append({
                'dataset'  : 'NIH',
                'model'    : arch_label,
                'condition': cond_key,
                'setting'  : 'blind',
                'mean'     : round(pt, 4),
                'ci_lo'    : round(lo, 4),
                'ci_hi'    : round(hi, 4),
            })
            del model
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"  {cond_key:6s}: ERROR — {e}")

# ── Save ───────────────────────────────────────────────────────────────────
blind_df = pd.DataFrame(results)
out_path = f"{SAVE_DIR}/nih_blind_oracle_ci.csv"
blind_df.to_csv(out_path, index=False)

print("\n=== NIH BLIND ORACLE CI ===")
print(blind_df.to_string(index=False))
print(f"\nSaved: {out_path}")

Device: cuda
NIH: torch.Size([3082, 1, 224, 224]), subjects=1739, n_classes=1739

NIH | ResNet-18 | Blind (trained on GS → test on raw)
  raw   : 73.32% [71.76, 74.91]  ← resnet18_Pretrained_raw_best.pth
  gs0   : 29.56% [27.93, 31.20]  ← resnet18_Pretrained_gs0_best.pth
  gs10  : 0.12% [0.00, 0.19]  ← resnet18_Pretrained_gs10_best.pth
  gs20  : 0.23% [0.09, 0.37]  ← resnet18_Pretrained_gs20_best.pth
  gs30  : 0.23% [0.09, 0.37]  ← resnet18_Pretrained_gs30_best.pth
  gs40  : 0.12% [0.00, 0.19]  ← resnet18_Pretrained_gs40_best.pth
  gs50  : 0.06% [0.00, 0.09]  ← resnet18_Pretrained_gs50_best.pth

NIH | DenseNet-121 | Blind (trained on GS → test on raw)
  raw   : 77.57% [76.10, 79.09]  ← densenet121_Pretrained_raw_best.pth
  gs0   : 22.77% [21.24, 24.28]  ← densenet121_Pretrained_gs0_best.pth
  gs10  : 0.58% [0.27, 0.82]  ← densenet121_Pretrained_gs10_best.pth
  gs20  : 0.23% [0.09, 0.37]  ← densenet121_Pretrained_gs20_best.pth
  gs30  : 0.29% [0.09, 0.46]  ← densenet121_Pretrained_gs30_

## Cell 2 — NIH Utility

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — NIH UTILITY CACHE INSPECTION (FIXED LOADER)
# ══════════════════════════════════════════════════════════════════════════════

import torch

CACHE_DIR = f"{BASE}/reid_chest-NIH/cache/nih_cxr/utility/tensors"

cache_files = sorted(glob.glob(f"{CACHE_DIR}/test_*.pt"))

print(f"Found {len(cache_files)} TEST cache files:\n")

for f in cache_files:
    size = os.path.getsize(f) / (1024**2)
    print(f"{size:6.2f} MB  {f}")

# ─────────────────────────────────────────────────────────────────────────────
# Inspect ONE file only (avoid heavy memory usage)
# ─────────────────────────────────────────────────────────────────────────────
f = cache_files[0]

print("\n" + "="*90)
print(f"INSPECTING: {f}")
print("="*90)

# 🔴 FIX: weights_only=False
data = torch.load(f, map_location="cpu", weights_only=False)

# ── Case 1: dict ─────────────────────────────────────────────────────────────
if isinstance(data, dict):
    print("\nType: dict")
    print("Keys:", list(data.keys()))

    for k, v in data.items():
        print(f"\nKey: {k}")
        print("  Type:", type(v))
        
        if isinstance(v, torch.Tensor):
            print("  Shape:", tuple(v.shape))
            print("  Sample:", v.flatten()[:5])
        
        elif isinstance(v, np.ndarray):
            print("  Shape:", v.shape)
            print("  Sample:", v.flatten()[:5])
        
        elif isinstance(v, list):
            print("  Length:", len(v))
            print("  Sample:", v[:3])
        
        else:
            print("  Value preview:", str(v)[:200])

# ── Case 2: tuple/list ───────────────────────────────────────────────────────
elif isinstance(data, (list, tuple)):
    print("\nType:", type(data))
    print("Length:", len(data))

    for i, v in enumerate(data[:5]):
        print(f"\nItem {i}: type={type(v)}")
        if hasattr(v, "shape"):
            print("  Shape:", v.shape)

else:
    print("\nUnknown structure:", type(data))

Found 7 TEST cache files:

1302.35 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/cache/nih_cxr/utility/tensors/test_gs0.pt
1302.35 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/cache/nih_cxr/utility/tensors/test_gs10.pt
1302.35 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/cache/nih_cxr/utility/tensors/test_gs20.pt
1302.35 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/cache/nih_cxr/utility/tensors/test_gs30.pt
1302.35 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/cache/nih_cxr/utility/tensors/test_gs40.pt
1302.35 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/cache/nih_cxr/utility/tensors/test_gs50.pt
1302.06 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/cache/nih_cxr/utility/tensors/test_raw.pt

INSPECTING: /home/jupyter/notebooks/reid_clean/reid_chest-NIH/cache/nih_cxr/utility/tensors/test_gs0.pt

Type: dict
Keys: ['x', 'y_int', 'y_mh', 'sids', 'genders']

Key: x
  Type: <class 'torch.Tensor'>
  Shape: (6802, 1, 224, 224)
  Sampl

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — FULL MODEL DIRECTORY INSPECTION (SAFE)
# ══════════════════════════════════════════════════════════════════════════════

MODEL_ROOT = f"{BASE}/reid_chest-NIH/models/nih_cxr/utility"

print("Scanning:", MODEL_ROOT)

all_files = []
for root, dirs, files in os.walk(MODEL_ROOT):
    for f in files:
        full_path = os.path.join(root, f)
        size = os.path.getsize(full_path) / (1024**2)
        all_files.append((full_path, size))

print(f"\nTotal files found: {len(all_files)}\n")

# sort by size (largest first → usually model files)
all_files_sorted = sorted(all_files, key=lambda x: -x[1])

print("=== Top 50 largest files ===")
for path, size in all_files_sorted[:50]:
    print(f"{size:8.2f} MB  {path}")

print("\n=== File extensions summary ===")
ext_count = {}
for path, _ in all_files:
    ext = os.path.splitext(path)[1]
    ext_count[ext] = ext_count.get(ext, 0) + 1

for k, v in sorted(ext_count.items()):
    print(f"{k or '[no ext]'}: {v}")

print("\n=== Directory structure (top levels) ===")
for root, dirs, files in os.walk(MODEL_ROOT):
    level = root.replace(MODEL_ROOT, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 3:
        continue
    for f in files[:5]:
        print(f"{indent}  {f}")

Scanning: /home/jupyter/notebooks/reid_clean/reid_chest-NIH/models/nih_cxr/utility

Total files found: 28

=== Top 50 largest files ===
   42.70 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/models/nih_cxr/utility/resnet18/scratch/gs30/best.pt
   42.70 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/models/nih_cxr/utility/resnet18/scratch/gs40/best.pt
   42.70 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/models/nih_cxr/utility/resnet18/scratch/gs0/best.pt
   42.70 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/models/nih_cxr/utility/resnet18/scratch/gs10/best.pt
   42.70 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/models/nih_cxr/utility/resnet18/scratch/raw/best.pt
   42.70 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/models/nih_cxr/utility/resnet18/scratch/gs50/best.pt
   42.70 MB  /home/jupyter/notebooks/reid_clean/reid_chest-NIH/models/nih_cxr/utility/resnet18/scratch/gs20/best.pt
   42.70 MB  /home/jupyter/notebooks/reid_clean/reid_c

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — MINIMAL MODEL FACTORY (INFERENCE ONLY)
# ══════════════════════════════════════════════════════════════════════════════

import torch.nn as nn
import torchvision.models as models

NUM_CLASSES = 11
DROPOUT_P = 0.5

def get_model(arch: str = 'resnet18', weights=None):
    is_pretrained = weights is not None

    if arch == 'resnet18':
        model = models.resnet18(weights=weights)

        # convert to 1-channel
        old_conv = model.conv1
        model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2,
                                padding=3, bias=False)

        if is_pretrained:
            with torch.no_grad():
                model.conv1.weight.copy_(
                    old_conv.weight.mean(dim=1, keepdim=True)
                )

        num_ftrs = model.fc.in_features
        head_linear = nn.Linear(num_ftrs, NUM_CLASSES)

        model.fc = nn.Sequential(
            nn.Dropout(p=DROPOUT_P),
            head_linear
        )

    elif arch == 'densenet121':
        model = models.densenet121(weights=weights,
                                   memory_efficient=True)

        # convert to 1-channel
        old_conv = model.features.conv0
        model.features.conv0 = nn.Conv2d(1, 64, kernel_size=7,
                                         stride=2, padding=3,
                                         bias=False)

        if is_pretrained:
            with torch.no_grad():
                model.features.conv0.weight.copy_(
                    old_conv.weight.mean(dim=1, keepdim=True)
                )

        num_ftrs = model.classifier.in_features
        head_linear = nn.Linear(num_ftrs, NUM_CLASSES)

        model.classifier = nn.Sequential(
            nn.Dropout(p=DROPOUT_P),
            head_linear
        )

    else:
        raise ValueError(f"Unknown architecture: {arch}")

    return model.to(DEVICE)

def aggregate_subject_probs(
    subject_ids,
    probs,
    labels_mh,
):
    """
    Aggregate image-level predictions to subject-level using mean pooling.
    Matches NIH utility pipeline exactly.
    """

    subj_to_probs  = {}
    subj_to_labels = {}

    for sid, p, y in zip(subject_ids, probs, labels_mh):
        sid = int(sid)
        subj_to_probs.setdefault(sid, []).append(p)
        subj_to_labels.setdefault(sid, []).append(y)

    subj_ids = np.array(list(subj_to_probs.keys()), dtype=np.int64)

    subj_probs = np.array(
        [np.mean(subj_to_probs[sid], axis=0) for sid in subj_ids],
        dtype=np.float32
    )

    subj_labels = np.array(
        [(np.mean(subj_to_labels[sid], axis=0) >= 0.5).astype(np.float32)
         for sid in subj_ids],
        dtype=np.float32
    )

    return subj_ids, subj_probs, subj_labels

In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — NIH UTILITY FINAL (INFERENCE + SUBJECT-LEVEL BOOTSTRAP)
# ══════════════════════════════════════════════════════════════════════════════

import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_ROOT = f"{BASE}/reid_chest-NIH/models/nih_cxr/utility"
TENSOR_DIR = f"{BASE}/reid_chest-NIH/cache/nih_cxr/utility/tensors"
SAVE_DIR   = f"{BASE}/CI_results"
os.makedirs(SAVE_DIR, exist_ok=True)

CONDITIONS = ["raw", "gs0", "gs10", "gs20", "gs30", "gs40", "gs50"]
ARCHS = ["resnet18", "densenet121"]
NUM_CLASSES = 11

# ─────────────────────────────────────────────────────────────────────────────
# Helper: load test tensors (your exact structure)
# ─────────────────────────────────────────────────────────────────────────────
def load_test(condition):
    obj = torch.load(f"{TENSOR_DIR}/test_{condition}.pt",
                     map_location="cpu", weights_only=False)

    X    = obj["x"]
    sids = obj["sids"]

    # Handle both formats (exactly like your pipeline)
    if "y_mh" in obj:
        y_mh = obj["y_mh"]
    elif "y_int" in obj:
        y_int = obj["y_int"].numpy()
        y_mh_np = np.zeros((len(y_int), NUM_CLASSES), dtype=np.float32)
        y_mh_np[np.arange(len(y_int)), y_int] = 1.0
        y_mh = torch.from_numpy(y_mh_np)
    elif "y" in obj:
        y_int = obj["y"].numpy()
        y_mh_np = np.zeros((len(y_int), NUM_CLASSES), dtype=np.float32)
        y_mh_np[np.arange(len(y_int)), y_int] = 1.0
        y_mh = torch.from_numpy(y_mh_np)
    else:
        raise KeyError("No label key found in tensor file")

    return X, y_mh, sids

# ─────────────────────────────────────────────────────────────────────────────
# Helper: inference → subject-level probs
# ─────────────────────────────────────────────────────────────────────────────
def run_inference(model, X, y_mh, sids):
    loader = DataLoader(TensorDataset(X, y_mh),
                        batch_size=64, shuffle=False)

    model.eval()
    probs = []

    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            probs.append(torch.sigmoid(logits).cpu().numpy())

    probs = np.concatenate(probs, axis=0)
    y_np  = y_mh.numpy()

    # subject aggregation (YOUR pipeline)
    subj_ids, subj_probs, subj_labels = aggregate_subject_probs(
        sids, probs, y_np
    )

    return subj_ids, subj_probs, subj_labels

# ─────────────────────────────────────────────────────────────────────────────
# Helper: compute macro AUC (same logic as compute_metrics)
# ─────────────────────────────────────────────────────────────────────────────
def compute_macro_auc(subj_probs, subj_labels):
    aucs = []
    for c in range(subj_probs.shape[1]):
        y_true = subj_labels[:, c]
        y_score = subj_probs[:, c]

        if y_true.sum() < 2 or (1 - y_true).sum() < 2:
            continue
        try:
            aucs.append(roc_auc_score(y_true, y_score))
        except:
            continue

    return float(np.mean(aucs)) if aucs else np.nan

# ─────────────────────────────────────────────────────────────────────────────
# Bootstrap (subject-level)
# ─────────────────────────────────────────────────────────────────────────────
def bootstrap_auc(subj_ids, subj_probs, subj_labels):
    rng = np.random.default_rng(RANDOM_SEED)
    n = len(subj_ids)

    aucs = []

    for _ in range(BOOTSTRAP_B):
        idx = rng.choice(n, size=n, replace=True)

        auc = compute_macro_auc(
            subj_probs[idx],
            subj_labels[idx]
        )
        if not np.isnan(auc):
            aucs.append(auc)

    aucs = np.array(aucs)

    return (
        aucs.mean(),
        np.percentile(aucs, CI_LOW),
        np.percentile(aucs, CI_HIGH)
    )

# ─────────────────────────────────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────────────────────────────────
results = []

for arch in ARCHS:
    print(f"\n=== {arch.upper()} ===")

    for cond in CONDITIONS:
        print(f"  → {cond}")

        # load data
        X, y_mh, sids = load_test(cond)

        # build model (SCRATCH ONLY)
        model = get_model(arch=arch, weights=None)
        model_path = f"{MODEL_ROOT}/{arch}/scratch/{cond}/best.pt"

        model.load_state_dict(
            torch.load(model_path, map_location=DEVICE, weights_only=True)
        )
        model.to(DEVICE)

        # inference
        subj_ids, subj_probs, subj_labels = run_inference(
            model, X, y_mh, sids
        )

        # bootstrap
        mean, lo, hi = bootstrap_auc(
            subj_ids, subj_probs, subj_labels
        )

        results.append({
            "arch": MODEL_RENAME[arch],
            "condition": cond,
            "auc_mean": mean,
            "ci_lower": lo,
            "ci_upper": hi,
            "n_subjects": len(subj_ids)
        })

        # cleanup
        del model, X, y_mh
        torch.cuda.empty_cache()

# ─────────────────────────────────────────────────────────────────────────────
# Save
# ─────────────────────────────────────────────────────────────────────────────
df = pd.DataFrame(results)

df["gs_pct"] = df["condition"].str.extract(r"(\d+)").fillna(0).astype(int)
df = df.sort_values(["arch", "gs_pct"]).drop(columns="gs_pct")

save_path = f"{SAVE_DIR}/nih_utility_ci.csv"
df.to_csv(save_path, index=False)

print("\n✓ NIH Utility CI DONE")
print("Saved to:", save_path)
display(df)


=== RESNET18 ===
  → raw
  → gs0
  → gs10
  → gs20
  → gs30
  → gs40
  → gs50

=== DENSENET121 ===
  → raw
  → gs0
  → gs10
  → gs20
  → gs30
  → gs40
  → gs50

✓ NIH Utility CI DONE
Saved to: /home/jupyter/notebooks/reid_clean/CI_results/nih_utility_ci.csv


,arch,condition,auc_mean,ci_lower,ci_upper,n_subjects
7,DenseNet-121,raw,0.743332,0.729544,0.756494,2699
8,DenseNet-121,gs0,0.751007,0.737905,0.764105,2699
9,DenseNet-121,gs10,0.738137,0.724296,0.751262,2699
10,DenseNet-121,gs20,0.735819,0.722431,0.748530,2699
11,DenseNet-121,gs30,0.732402,0.718778,0.745278,2699
12,DenseNet-121,gs40,0.723891,0.710493,0.737907,2699
13,DenseNet-121,gs50,0.700023,0.686893,0.713238,2699
0,ResNet-18,raw,0.743329,0.730818,0.756041,2699
1,ResNet-18,gs0,0.734320,0.721430,0.746958,2699
2,ResNet-18,gs10,0.725973,0.712386,0.738985,2699


## NIH FEATURE RE-ID CI SETUP

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — NIH FEATURE RE-ID CI SETUP
# pretrained + adaptive + test split + subject level only
# ══════════════════════════════════════════════════════════════════════════════

import gc
import math
import random
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_ROOT = f"{BASE}/reid_chest-NIH/models/nih_cxr/utility"
TENSOR_DIR = f"{BASE}/reid_chest-NIH/cache/nih_cxr/utility/tensors"
SAVE_DIR   = f"{BASE}/CI_results"
os.makedirs(SAVE_DIR, exist_ok=True)

NUM_CLASSES = 11
DROPOUT_P = 0.5
BATCH_SIZE_EMB = 128
GLOBAL_SEED = 1337

FE_ARCHS = ["resnet18", "densenet121"]
FE_CONDITIONS = ["raw", "gs0", "gs10", "gs20", "gs30", "gs40", "gs50"]
FE_K_GALLERY = [1, 5]
FE_K_PROBE = 1

def hard_fail(msg):
    raise RuntimeError(f"[HARD FAIL] {msg}")

def strip_prefix(sd):
    if any(k.startswith("module.") for k in sd):
        return {k.replace("module.", "", 1): v for k, v in sd.items()}
    return sd

def looks_like_state_dict(d):
    return (
        isinstance(d, dict)
        and len(d) > 0
        and all(isinstance(k, str) and torch.is_tensor(v)
                for k, v in list(d.items())[:10])
    )

def build_model_like_utility(arch):
    if arch == "resnet18":
        model = tvm.resnet18(weights=None)
        model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        model.fc = nn.Sequential(
            nn.Dropout(p=DROPOUT_P),
            nn.Linear(model.fc.in_features, NUM_CLASSES)
        )
        return model

    if arch == "densenet121":
        model = tvm.densenet121(weights=None, memory_efficient=True)
        model.features.conv0 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        model.classifier = nn.Sequential(
            nn.Dropout(p=DROPOUT_P),
            nn.Linear(model.classifier.in_features, NUM_CLASSES)
        )
        return model

    hard_fail(f"Unknown arch: {arch}")

def load_utility_model(arch, condition):
    path = f"{MODEL_ROOT}/{arch}/pretrained/{condition}/best.pt"
    if not os.path.exists(path):
        hard_fail(f"Missing model: {path}")

    obj = torch.load(path, map_location="cpu", weights_only=True)
    sd = strip_prefix(obj if looks_like_state_dict(obj) else obj.get("state_dict", obj))

    model = build_model_like_utility(arch)
    model.load_state_dict(sd, strict=True)
    model.eval().to(DEVICE)
    return model

def build_penultimate_extractor(model, arch):
    if arch == "resnet18":
        class ResNet18Embedder(nn.Module):
            def __init__(self, m):
                super().__init__()
                self.m = m
            def forward(self, x):
                x = self.m.conv1(x)
                x = self.m.bn1(x)
                x = self.m.relu(x)
                x = self.m.maxpool(x)
                x = self.m.layer1(x)
                x = self.m.layer2(x)
                x = self.m.layer3(x)
                x = self.m.layer4(x)
                x = self.m.avgpool(x)
                return torch.flatten(x, 1)
        return ResNet18Embedder(model).eval().to(DEVICE)

    if arch == "densenet121":
        class DenseNet121Embedder(nn.Module):
            def __init__(self, m):
                super().__init__()
                self.m = m
            def forward(self, x):
                x = self.m.features(x)
                x = F.relu(x, inplace=False)
                x = F.adaptive_avg_pool2d(x, (1, 1))
                return torch.flatten(x, 1)
        return DenseNet121Embedder(model).eval().to(DEVICE)

    hard_fail(f"Unknown arch: {arch}")

@torch.no_grad()
def extract_embeddings(x, embedder):
    outs = []
    for i in range(0, x.shape[0], BATCH_SIZE_EMB):
        xb = x[i:i+BATCH_SIZE_EMB].to(DEVICE, non_blocking=True)
        eb = embedder(xb).float()
        eb = F.normalize(eb, p=2, dim=1)
        outs.append(eb.cpu())
    return torch.cat(outs, dim=0)

def load_test_tensor(condition):
    obj = torch.load(
        f"{TENSOR_DIR}/test_{condition}.pt",
        map_location="cpu",
        weights_only=False
    )
    return obj["x"], np.asarray(obj["sids"])

def build_gallery_probe_from_sids(sids, k_gallery, k_probe=1):
    groups = defaultdict(list)
    for i, sid in enumerate(sids):
        groups[int(sid)].append(i)

    gallery_idx, gallery_sid = [], []
    probe_idx, probe_sid = [], []

    for pid, idxs in sorted(groups.items()):
        if len(idxs) < 2:
            continue

        r = random.Random(hash((GLOBAL_SEED, pid)) & 0xFFFFFFFF)
        idxs_shuf = idxs.copy()
        r.shuffle(idxs_shuf)

        n = len(idxs_shuf)
        n_g = max(int(math.floor(n * 0.5)), 1)
        n_g = min(n_g, n - 1)

        g = idxs_shuf[:n_g]
        p = idxs_shuf[n_g:]

        g = g[:k_gallery]
        p = p[:k_probe]

        if len(g) < 1 or len(p) < 1:
            continue

        gallery_idx.extend(g)
        gallery_sid.extend([str(pid)] * len(g))
        probe_idx.extend(p)
        probe_sid.extend([str(pid)] * len(p))

    return gallery_idx, gallery_sid, probe_idx, probe_sid

def mean_embeddings_by_subject(emb, sid_list):
    groups = defaultdict(list)
    for i, s in enumerate(sid_list):
        groups[s].append(i)

    subj_ids = sorted(groups.keys())
    subj_embs = []

    for s in subj_ids:
        idx = groups[s]
        m = emb[idx].mean(dim=0, keepdim=True)
        m = F.normalize(m, p=2, dim=1)
        subj_embs.append(m)

    return torch.cat(subj_embs, dim=0), subj_ids

def subject_ranks(probe_emb, probe_sid, gallery_emb, gallery_sid):
    sim = (probe_emb @ gallery_emb.T).cpu().numpy()
    gallery_sid = np.asarray(gallery_sid)
    probe_sid = np.asarray(probe_sid)

    ranks = []

    for i in range(sim.shape[0]):
        order = np.argsort(-sim[i])
        ranked_sids = gallery_sid[order]
        pos = np.where(ranked_sids == probe_sid[i])[0]
        if len(pos) == 0:
            hard_fail(f"Probe subject {probe_sid[i]} missing from gallery")
        ranks.append(int(pos[0]) + 1)

    return np.asarray(ranks), probe_sid

def bootstrap_rank_metrics(ranks, B=BOOTSTRAP_B, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    n = len(ranks)

    r1_vals = []
    r5_vals = []

    for _ in range(B):
        idx = rng.choice(n, size=n, replace=True)
        r = ranks[idx]
        r1_vals.append(np.mean(r == 1))
        r5_vals.append(np.mean(r <= 5))

    r1_vals = np.asarray(r1_vals)
    r5_vals = np.asarray(r5_vals)

    return {
        "rank1_mean": float(np.mean(ranks == 1)),
        "rank1_ci_lower": float(np.percentile(r1_vals, CI_LOW)),
        "rank1_ci_upper": float(np.percentile(r1_vals, CI_HIGH)),
        "rank5_mean": float(np.mean(ranks <= 5)),
        "rank5_ci_lower": float(np.percentile(r5_vals, CI_LOW)),
        "rank5_ci_upper": float(np.percentile(r5_vals, CI_HIGH)),
    }

print("✓ Cell 6 ready")

✓ Cell 6 ready


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — NIH FEATURE RE-ID CI COMPUTATION
# pretrained + adaptive + test split + subject level only
# ══════════════════════════════════════════════════════════════════════════════

rows = []

for arch in FE_ARCHS:
    print(f"\n=== {arch.upper()} | PRETRAINED ===")

    for cond in FE_CONDITIONS:
        print(f"  → condition: {cond}")

        X, sids = load_test_tensor(cond)

        model = load_utility_model(arch, cond)
        embedder = build_penultimate_extractor(model, arch)

        for k_gallery in FE_K_GALLERY:
            print(f"     k_gallery={k_gallery}, k_probe=1")

            g_idx, g_sid, p_idx, p_sid = build_gallery_probe_from_sids(
                sids=sids,
                k_gallery=k_gallery,
                k_probe=FE_K_PROBE
            )

            g_x = X[g_idx]
            p_x = X[p_idx]

            g_emb = extract_embeddings(g_x, embedder)
            p_emb = extract_embeddings(p_x, embedder)

            g_subj_emb, g_subj_ids = mean_embeddings_by_subject(g_emb, g_sid)
            p_subj_emb, p_subj_ids = mean_embeddings_by_subject(p_emb, p_sid)

            inter = sorted(set(g_subj_ids) & set(p_subj_ids))
            g_map = {s: i for i, s in enumerate(g_subj_ids)}
            p_map = {s: i for i, s in enumerate(p_subj_ids)}

            g_keep = [g_map[s] for s in inter]
            p_keep = [p_map[s] for s in inter]

            ranks, rank_sids = subject_ranks(
                probe_emb=p_subj_emb[p_keep],
                probe_sid=inter,
                gallery_emb=g_subj_emb[g_keep],
                gallery_sid=inter
            )

            out = bootstrap_rank_metrics(ranks)

            rows.append({
                "dataset": "NIH",
                "arch": MODEL_RENAME[arch],
                "init": "pretrained",
                "attacker_type": "adaptive",
                "eval_split": "test",
                "level": "subject_level",
                "condition": cond,
                "k_gallery": k_gallery,
                "k_probe": FE_K_PROBE,
                "n_subjects": len(inter),
                **out
            })

            del g_x, p_x, g_emb, p_emb, g_subj_emb, p_subj_emb
            torch.cuda.empty_cache()
            gc.collect()

        del model, embedder, X
        torch.cuda.empty_cache()
        gc.collect()

nih_fe_ci = pd.DataFrame(rows)

nih_fe_ci["gs_pct"] = (
    nih_fe_ci["condition"].str.extract(r"(\d+)").fillna(0).astype(int)
)
nih_fe_ci = nih_fe_ci.sort_values(
    ["arch", "k_gallery", "gs_pct"]
).drop(columns="gs_pct")

save_path = f"{SAVE_DIR}/nih_fe_reid_ci.csv"
nih_fe_ci.to_csv(save_path, index=False)

print("\n✓ NIH FE Re-ID CI DONE")
print("Saved to:", save_path)
display(nih_fe_ci)


=== RESNET18 | PRETRAINED ===
  → condition: raw
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs0
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs10
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs20
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs30
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs40
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs50
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1

=== DENSENET121 | PRETRAINED ===
  → condition: raw
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs0
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs10
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs20
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs30
     k_gallery=1, k_probe=1
     k_gallery=5, k_probe=1
  → condition: gs40
     k_

,dataset,arch,init,attacker_type,eval_split,level,condition,k_gallery,k_probe,n_subjects,rank1_mean,rank1_ci_lower,rank1_ci_upper,rank5_mean,rank5_ci_lower,rank5_ci_upper
14,NIH,DenseNet-121,pretrained,adaptive,test,subject_level,raw,1,1,1060,0.268868,0.242453,0.295283,0.383962,0.353774,0.414151
16,NIH,DenseNet-121,pretrained,adaptive,test,subject_level,gs0,1,1,1060,0.163208,0.141509,0.185873,0.281132,0.254717,0.307547
18,NIH,DenseNet-121,pretrained,adaptive,test,subject_level,gs10,1,1,1060,0.092453,0.075472,0.110377,0.180189,0.156604,0.204717
20,NIH,DenseNet-121,pretrained,adaptive,test,subject_level,gs20,1,1,1060,0.066038,0.051887,0.081132,0.157547,0.135849,0.180189
22,NIH,DenseNet-121,pretrained,adaptive,test,subject_level,gs30,1,1,1060,0.035849,0.025472,0.047170,0.099057,0.081132,0.117005
24,NIH,DenseNet-121,pretrained,adaptive,test,subject_level,gs40,1,1,1060,0.031132,0.020755,0.041509,0.103774,0.085849,0.121698
26,NIH,DenseNet-121,pretrained,adaptive,test,subject_level,gs50,1,1,1060,0.030189,0.019811,0.041509,0.079245,0.064151,0.095283
15,NIH,DenseNet-121,pretrained,adaptive,test,subject_level,raw,5,1,1060,0.264151,0.237736,0.291509,0.399057,0.369811,0.430189
17,NIH,DenseNet-121,pretrained,adaptive,test,subject_level,gs0,5,1,1060,0.169811,0.148113,0.193396,0.300943,0.273585,0.329245
19,NIH,DenseNet-121,pretrained,adaptive,test,subject_level,gs10,5,1,1060,0.106604,0.087736,0.125472,0.210377,0.186792,0.234929


## NIH UNet Reconstructions

In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# RECONSTRUCTION CI — REQUIRED FUNCTIONS (SELF-CONTAINED)
# ══════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader

# ─────────────────────────────────────────────────────────────────────────────
# SSIM (same as pipeline)
# ─────────────────────────────────────────────────────────────────────────────
def _gaussian_window(window_size=11, sigma=1.5, channels=1, device='cpu'):
    coords = torch.arange(window_size, device=device).float() - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w = (g[:, None] * g[None, :]).unsqueeze(0).unsqueeze(0)
    return w.repeat(channels, 1, 1, 1)

def ssim_torch(x, y, window_size=11, sigma=1.5, data_range=1.0, eps=1e-6):
    x, y = x.float(), y.float()
    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2

    ch = x.shape[1]
    w = _gaussian_window(window_size, sigma, ch, device=x.device)

    pad = window_size // 2
    mu_x = F.conv2d(x, w, padding=pad, groups=ch)
    mu_y = F.conv2d(y, w, padding=pad, groups=ch)

    sigma_x2 = F.conv2d(x * x, w, padding=pad, groups=ch) - mu_x ** 2
    sigma_y2 = F.conv2d(y * y, w, padding=pad, groups=ch) - mu_y ** 2
    sigma_xy = F.conv2d(x * y, w, padding=pad, groups=ch) - mu_x * mu_y

    num = (2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)
    den = (mu_x**2 + mu_y**2 + C1) * (sigma_x2 + sigma_y2 + C2)

    return (num / torch.clamp(den, min=eps)).mean(dim=(1, 2, 3))


# ─────────────────────────────────────────────────────────────────────────────
# PSNR
# ─────────────────────────────────────────────────────────────────────────────
def psnr_torch(x, y, data_range=1.0, eps=1e-8):
    mse = F.mse_loss(x, y, reduction='none').mean(dim=(1, 2, 3))
    return 10.0 * torch.log10((data_range**2) / (mse + eps))


# ─────────────────────────────────────────────────────────────────────────────
# Adaptive normalization dataset (for ReID)
# ─────────────────────────────────────────────────────────────────────────────
class AdaptiveNormDataset(Dataset):
    def __init__(self, imgs, labels, mean, std):
        self.imgs = imgs
        self.labels = labels
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx].copy()
        img = (img - self.mean) / (self.std + 1e-8)
        return (
            torch.tensor(img).unsqueeze(0).float(),
            torch.tensor(int(self.labels[idx]), dtype=torch.long),
        )


print("✓ Reconstruction CI dependencies ready")

✓ Reconstruction CI dependencies ready


In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD NIH REID MODELS (from reconstruction pipeline)
# ══════════════════════════════════════════════════════════════════════════════

import torch.nn as nn
from torchvision import models

REID_MODEL_DIR = f"{BASE}/reid_chest-NIH/models/nih_cxr/reid"
NUM_CLASSES_REID = 1739  # from your pipeline

def load_reid_model(arch: str, init: str):
    fname = f"{arch}_{init}_raw_best.pth"
    path  = f"{REID_MODEL_DIR}/{arch}/{fname}"

    if not os.path.exists(path):
        raise FileNotFoundError(f"ReID model not found: {path}")

    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    sd   = ckpt.get('state_dict', ckpt)

    if arch == 'resnet18':
        has_seq = 'fc.1.weight' in sd

        model = models.resnet18(weights=None)
        model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        if has_seq:
            model.fc = nn.Sequential(
                nn.Dropout(p=0.0),
                nn.Linear(512, NUM_CLASSES_REID)
            )
        else:
            model.fc = nn.Linear(512, NUM_CLASSES_REID)

    else:  # densenet121
        has_seq = 'classifier.1.weight' in sd

        model = models.densenet121(weights=None)
        model.features[0] = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        if has_seq:
            model.classifier = nn.Sequential(
                nn.Dropout(p=0.0),
                nn.Linear(1024, NUM_CLASSES_REID)
            )
        else:
            model.classifier = nn.Linear(1024, NUM_CLASSES_REID)

    model.load_state_dict(
        {k.replace('module.', ''): v for k, v in sd.items()}
    )

    model = model.to(DEVICE)
    model.eval()

    print(f"✓ Loaded {arch} {init}")
    return model

In [26]:
# ══════════════════════════════════════════════════════════════════════════════
# NIH RECONSTRUCTION CI (SSIM, PSNR, ReID) — FIXED
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

BOOTSTRAP_B = 2000
CI_LOW = 2.5
CI_HIGH = 97.5
RANDOM_SEED = 42

REID_NPZ_DIR = Path(f"{BASE}/reid_chest-NIH/data/chest/reid_npz")

def load_reid_npz(split, condition):
    path = REID_NPZ_DIR / f"reid_{split}_{condition}.npz"
    if not path.exists():
        raise FileNotFoundError(f"NPZ not found: {path}")
    data = np.load(path)
    images = data["images"].astype(np.float32)
    labels = data["labels"].astype(np.int64)
    pids   = data["patient_ids"].astype(np.int64)
    data.close()
    return images, labels, pids

# ── Load raw reference ───────────────────────────────────────────────────────
test_raw, test_labels, test_pids = load_reid_npz("test", "raw")

print("Loaded NIH test raw:")
print("  images:", test_raw.shape)
print("  labels:", test_labels.shape)
print("  unique patients:", len(np.unique(test_pids)))

RESULTS_DIR = f"{BASE}/reid_chest-NIH/results/nih_cxr/unet"
RECON_TAGS = ["gs0", "gs50"]

def bootstrap_ci(vals):
    rng = np.random.default_rng(RANDOM_SEED)
    n = len(vals)
    samples = []

    for _ in range(BOOTSTRAP_B):
        idx = rng.choice(n, size=n, replace=True)
        samples.append(np.mean(vals[idx]))

    samples = np.array(samples)

    return (
        np.mean(vals),
        np.percentile(samples, CI_LOW),
        np.percentile(samples, CI_HIGH)
    )

raw_np = test_raw.copy()
results = []

for tag in RECON_TAGS:
    print(f"\n=== {tag.upper()} ===")

    # ── Load recon ───────────────────────────────────────────────────────────
    recon_path = f"{RESULTS_DIR}/recon_{tag}_test.pt"
    recon = torch.load(recon_path, map_location="cpu")

    if recon.ndim == 3:
        recon = recon.unsqueeze(1)

    raw_t = torch.from_numpy(raw_np)
    if raw_t.ndim == 3:
        raw_t = raw_t.unsqueeze(1)

    # ── INTENSITY ────────────────────────────────────────────────────────────
    ssim_vals = ssim_torch(recon, raw_t).numpy()
    psnr_vals = psnr_torch(recon, raw_t).numpy()

    ssim_mean, ssim_lo, ssim_hi = bootstrap_ci(ssim_vals)
    psnr_mean, psnr_lo, psnr_hi = bootstrap_ci(psnr_vals)

    print(f"SSIM: {ssim_mean:.4f} [{ssim_lo:.4f}, {ssim_hi:.4f}]")
    print(f"PSNR: {psnr_mean:.2f} [{psnr_lo:.2f}, {psnr_hi:.2f}]")

    # ── FUNCTIONALITY (ReID CI) ───────────────────────────────────────────────
    for arch in ["resnet18", "densenet121"]:

        model = load_reid_model(arch, "Pretrained")

        # FIX: remove channel before dataset
        recon_np = recon.squeeze(1).numpy()

        mean = float(recon_np.mean())
        std  = float(recon_np.std())

        ds = AdaptiveNormDataset(
            recon_np,
            test_labels,
            mean,
            std
        )

        dl = DataLoader(ds, batch_size=32, shuffle=False)

        correct_top1 = []
        correct_top5 = []

        with torch.no_grad():
            for xb, yb in dl:
                xb = xb.to(DEVICE)
                logits = model(xb)
                probs = torch.softmax(logits, dim=1)

                preds = logits.argmax(dim=1).cpu().numpy()
                y_np  = yb.numpy()
                prob_np = probs.cpu().numpy()

                correct_top1.extend((preds == y_np).astype(int))

                for i in range(len(y_np)):
                    top5 = np.argsort(prob_np[i])[-5:]
                    correct_top5.append(int(y_np[i] in top5))

        correct_top1 = np.array(correct_top1)
        correct_top5 = np.array(correct_top5)

        t1_mean, t1_lo, t1_hi = bootstrap_ci(correct_top1)
        t5_mean, t5_lo, t5_hi = bootstrap_ci(correct_top5)

        print(f"{arch} Top1: {t1_mean:.4f} [{t1_lo:.4f}, {t1_hi:.4f}]")
        print(f"{arch} Top5: {t5_mean:.4f} [{t5_lo:.4f}, {t5_hi:.4f}]")

        results.append({
            "dataset": "NIH",
            "condition": tag,
            "arch": MODEL_RENAME[arch],
            "ssim_mean": ssim_mean,
            "ssim_ci_lower": ssim_lo,
            "ssim_ci_upper": ssim_hi,
            "psnr_mean": psnr_mean,
            "psnr_ci_lower": psnr_lo,
            "psnr_ci_upper": psnr_hi,
            "rank1_mean": t1_mean,
            "rank1_ci_lower": t1_lo,
            "rank1_ci_upper": t1_hi,
            "rank5_mean": t5_mean,
            "rank5_ci_lower": t5_lo,
            "rank5_ci_upper": t5_hi,
        })

        del model
        torch.cuda.empty_cache()

# ── Save ─────────────────────────────────────────────────────────────────────
df_recon = pd.DataFrame(results)
save_path = f"{BASE}/CI_results/nih_recon_ci.csv"
df_recon.to_csv(save_path, index=False)

print("\n✓ NIH Reconstruction CI DONE")
print("Saved to:", save_path)
display(df_recon)

Loaded NIH test raw:
  images: (3082, 224, 224)
  labels: (3082,)
  unique patients: 1739

=== GS0 ===
SSIM: 0.9189 [0.9170, 0.9206]
PSNR: 30.46 [30.28, 30.65]
✓ Loaded resnet18 Pretrained
resnet18 Top1: 0.2271 [0.2128, 0.2417]
resnet18 Top5: 0.4082 [0.3907, 0.4250]
✓ Loaded densenet121 Pretrained
densenet121 Top1: 0.1450 [0.1327, 0.1577]
densenet121 Top5: 0.2829 [0.2667, 0.2995]

=== GS50 ===
SSIM: 0.7369 [0.7342, 0.7395]
PSNR: 22.30 [22.22, 22.39]
✓ Loaded resnet18 Pretrained
resnet18 Top1: 0.0016 [0.0003, 0.0032]
resnet18 Top5: 0.0088 [0.0058, 0.0123]
✓ Loaded densenet121 Pretrained
densenet121 Top1: 0.0019 [0.0006, 0.0036]
densenet121 Top5: 0.0094 [0.0062, 0.0130]

✓ NIH Reconstruction CI DONE
Saved to: /home/jupyter/notebooks/reid_clean/CI_results/nih_recon_ci.csv


,dataset,condition,arch,ssim_mean,ssim_ci_lower,ssim_ci_upper,psnr_mean,psnr_ci_lower,psnr_ci_upper,rank1_mean,rank1_ci_lower,rank1_ci_upper,rank5_mean,rank5_ci_lower,rank5_ci_upper
0,NIH,gs0,ResNet-18,0.918862,0.916989,0.920637,30.463289,30.280272,30.646027,0.227125,0.212849,0.241734,0.408177,0.390655,0.425049
1,NIH,gs0,DenseNet-121,0.918862,0.916989,0.920637,30.463289,30.280272,30.646027,0.145036,0.132706,0.157690,0.282933,0.266702,0.299489
2,NIH,gs50,ResNet-18,0.736923,0.734214,0.739487,22.301676,22.220232,22.387335,0.001622,0.000324,0.003245,0.008761,0.005840,0.012330
3,NIH,gs50,DenseNet-121,0.736923,0.734214,0.739487,22.301676,22.220232,22.387335,0.001947,0.000649,0.003569,0.009409,0.006165,0.012979
